In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

RESULTS_ROOT = Path(r"D:\masteruwefduyqeahfdqe\ASR-project")

filename_pattern = re.compile(
    r"^(mono|multi|cross)_(english|greek|mandarin)_(.*?)_metrics_new\.csv$",
    flags=re.IGNORECASE,
)

loaded_results = []

for path in RESULTS_ROOT.rglob("*.csv"):
    match = filename_pattern.match(path.name)

    if match:
        strategy, language, feature_set = match.groups()

        # Load all model results from the CSV
        df = pd.read_csv(path)

        # Add metadata from the filename
        df["strategy"] = strategy.lower()
        df["language"] = language.lower()
        df["feature_set"] = feature_set
        df["source_file"] = path.name
        df["source_path"] = str(path)

        loaded_results.append(df)

        print(f"Loaded {path.name}: {df.shape}")

In [ ]:
# concatenating all the loaded results into a single DataFrame
results_df = pd.concat(
    loaded_results,
    axis=0,
    join="outer",
    ignore_index=True,
)


In [ ]:
display(results_df)
if_results_df = results_df

In [ ]:
NON_INTERPRETABLE_MONO_FILE = (
    RESULTS_ROOT
    / r"RESULTS/none_interpretable_model_results_monolingual_train_test_all_features_and_isolated.csv"
)
NON_INTERPRETABLE_CROSS_FILE = (
    RESULTS_ROOT
    / r"RESULTS/none_interpretable_model_results_crosslingual_train_two_test_heldout_language.csv"
)

NON_INTERPRETABLE_MULTI_FILE = (
    RESULTS_ROOT
    / r"RESULTS/none_interpretable_model_results_multilingual_train_all_test_each_language.csv"
)
non_df_mono = pd.read_csv(NON_INTERPRETABLE_MONO_FILE)
non_df_cross = pd.read_csv(NON_INTERPRETABLE_CROSS_FILE)
non_df_multi = pd.read_csv(NON_INTERPRETABLE_MULTI_FILE)
display(non_df_mono)

In [7]:
if_mono= if_results_df[if_results_df["strategy"] == "mono"]
if_cross= if_results_df[if_results_df["strategy"] == "cross"]
if_multi= if_results_df[if_results_df["strategy"] == "multi"]

## tables 

In [ ]:
METRIC_COLUMNS = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "auc",
]

FEATURE_NAME_MAPPING = {
    # Interpretable features
    "all_features": "All interpretable features",
    "cha_only_features": "Linguistic features",
    "wav_only_features": "Acoustic features",
    "subset_cha_features": "Selected linguistic features",
    "subset_wav_features": "Selected acoustic features",

    # Non-interpretable features
    "hubert": "HuBERT",
    "wav2vec2": "Wav2Vec2",
    "trillsson": "TRILLsson",
    "xvector": "x-vector",
    "x-vector": "x-vector",
    "all_features_fused": "All features fused",
}

LANGUAGE_ORDER = [
    "English",
    "Greek",
    "Mandarin",
]

FEATURE_ORDER = [
    "All interpretable features",
    "Acoustic features",
    "Linguistic features",
    "Selected acoustic features",
    "Selected linguistic features",
    "HuBERT",
    "Wav2Vec2",
    "TRILLsson",
    "x-vector",
    "All features fused",
]

def prepare_interpretable_results(df, strategy):
    result = df.copy()

    result["strategy"] = strategy

    result["language"] = (
        result["language"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.title()
    )

    result["features"] = (
        result["feature_set"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(FEATURE_NAME_MAPPING)
        .fillna(result["feature_set"])
    )

    # Interpretable results dont do PCA
    result["model_display"] = result["model"].astype(str)

    for metric in METRIC_COLUMNS:
        result[metric] = pd.to_numeric(
            result[metric],
            errors="coerce",
        )

    return result[
        [
            "strategy",
            "language",
            "features",
            "model_display",
            *METRIC_COLUMNS,
        ]
    ]

def prepare_non_interpretable_results(df, strategy):
    """Prepare non-interpretable results for analysis.
        Arg:
            df (pd.DataFrame: DataFrame containing non-interpretable results
            strategy (str):Strategy name 
        Returs:
            pd.DataFrame: Prepared DataFrame with standardized columns and metrics
         """

    result = df.copy()

    result["strategy"] = strategy

    result["language"] = (
        result["language"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.title()
    )

    result["features"] = (
        result["feature_set"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(FEATURE_NAME_MAPPING)
        .fillna(result["feature_set"])
    )

    for metric in METRIC_COLUMNS:
        result[metric] = pd.to_numeric(
            result[metric],
            errors="coerce",
        )

    # Removing the  rows where modle evaluation failed
    if "error" in result.columns:
        valid_rows = (
            result["error"].isna()
            | result["error"].astype(str).str.strip().eq("")
        )
        result = result.loc[valid_rows].copy()

    # Standardize PCA key, column titles, we were not consistent in the naming
    if "use_pca" in result.columns:
        result["use_pca_clean"] = (
            result["use_pca"]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(["true", "pca"])
        )

        result["pca_label"] = np.where(
            result["use_pca_clean"],
            "PCA",
            "No PCA",
        )

        result["model_display"] = (
            result["model"].astype(str)
            + " ("
            + result["pca_label"]
            + ")"
        )

    else:
        result["model_display"] = result["model"].astype(str)

    return result[
        ["strategy","language", "features", "model_display", *METRIC_COLUMNS ]
    ]


# selecting best rows wih best balanced accuracy for each language and feature set
def select_best_configurations(
    interpretable_df,
    non_interpretable_df,
    strategy,
):
    interpretable = prepare_interpretable_results(
        interpretable_df,
        strategy,
    )

    non_interpretable = prepare_non_interpretable_results(
        non_interpretable_df,
        strategy,
    )

    combined = pd.concat(
        [interpretable,non_interpretable,
        ],
        ignore_index=True,
        sort=False,
    )

    # Balanced accuracy selection criterion.
    combined = combined.dropna(
        subset=["balanced_accuracy"]
    )

   
    best = (
        combined
        .sort_values(
            by=[ "language","features","balanced_accuracy","f1","auc","accuracy","model_display",
            ],
            ascending=[True, True, False,
                False,
                False,


                False,
                True,
            ],
            na_position="last",
        )
        .groupby(
            [
                "language",
                "features",
            ],
            as_index=False,
            dropna=False,
        )
        .head(1)
        .reset_index(drop=True)
    )

    # All metrics here are taken from the same row that we selected
    best[METRIC_COLUMNS] = best[METRIC_COLUMNS].round(3)

    return best



# big results tables per participant per strategy
best_mono = select_best_configurations(
    interpretable_df=if_mono,
    non_interpretable_df=non_df_mono,
    strategy="mono",
)

best_cross = select_best_configurations(
    interpretable_df=if_cross,
    non_interpretable_df=non_df_cross,
    strategy="cross",
)

best_multi = select_best_configurations(
    interpretable_df=if_multi,
    non_interpretable_df=non_df_multi,
    strategy="multi",
)

display(best_mono)
display(best_cross)
display(best_multi)

,strategy,language,features,model_display,accuracy,balanced_accuracy,precision,recall,f1,auc
0,mono,English,Acoustic features,SVM-RBF,0.616,0.626,0.707,0.482,0.573,0.614
1,mono,English,All features fused,SVM_RBF (PCA),0.742,0.739,0.744,0.788,0.766,0.826
2,mono,English,All interpretable features,SVM-Linear,0.686,0.688,0.727,0.659,0.691,0.751
3,mono,English,HuBERT,MLP (No PCA),0.761,0.754,0.737,0.859,0.793,0.829
4,mono,English,Linguistic features,SVM-Linear,0.711,0.713,0.753,0.682,0.716,0.729
5,mono,English,Selected acoustic features,SVM-Linear,0.572,0.583,0.649,0.435,0.521,0.626
6,mono,English,Selected linguistic features,SVM-Linear,0.673,0.675,0.714,0.647,0.679,0.739
7,mono,English,TRILLsson,MLP (No PCA),0.780,0.773,0.755,0.871,0.809,0.827
8,mono,English,Wav2Vec2,SVM_RBF (No PCA),0.711,0.706,0.710,0.776,0.742,0.756
9,mono,English,x-vector,SVM_RBF (No PCA),0.667,0.660,0.667,0.753,0.707,0.744


,strategy,language,features,model_display,accuracy,balanced_accuracy,precision,recall,f1,auc
0,cross,English,Acoustic features,SVM-Linear,0.546,0.562,0.641,0.434,0.517,0.568
1,cross,English,All features fused,KNN (PCA),0.610,0.598,0.606,0.776,0.680,0.591
2,cross,English,All interpretable features,SVM-Linear,0.615,0.600,0.638,0.725,0.679,0.658
3,cross,English,HuBERT,MLP (PCA),0.597,0.594,0.618,0.647,0.632,0.618
4,cross,English,Linguistic features,SVM-RBF,0.592,0.556,0.595,0.848,0.700,0.643
5,cross,English,Selected acoustic features,KNN,0.530,0.516,0.574,0.628,0.600,0.510
6,cross,English,Selected linguistic features,KNN,0.593,0.586,0.634,0.650,0.642,0.615
7,cross,English,TRILLsson,SVM_Linear (No PCA),0.610,0.587,0.586,0.918,0.716,0.591
8,cross,English,Wav2Vec2,KNN (PCA),0.604,0.601,0.625,0.647,0.636,0.623
9,cross,English,x-vector,SVM_Linear (No PCA),0.572,0.581,0.639,0.459,0.534,0.568


,strategy,language,features,model_display,accuracy,balanced_accuracy,precision,recall,f1,auc
0,multi,English,Acoustic features,SVM-RBF,0.566,0.572,0.621,0.482,0.543,0.591
1,multi,English,All features fused,SVM_Linear (No PCA),0.723,0.717,0.711,0.812,0.758,0.809
2,multi,English,All interpretable features,SVM-Linear,0.648,0.650,0.693,0.612,0.650,0.712
3,multi,English,HuBERT,MLP (No PCA),0.767,0.760,0.740,0.871,0.800,0.823
4,multi,English,Linguistic features,SVM-Linear,0.616,0.608,0.620,0.729,0.670,0.726
5,multi,English,Selected acoustic features,SVM-Linear,0.560,0.554,0.581,0.635,0.607,0.576
6,multi,English,Selected linguistic features,SVM-RBF,0.610,0.611,0.646,0.600,0.622,0.639
7,multi,English,TRILLsson,SVM_RBF (No PCA),0.761,0.756,0.753,0.824,0.787,0.821
8,multi,English,Wav2Vec2,XGBoost (No PCA),0.660,0.652,0.653,0.776,0.710,0.720
9,multi,English,x-vector,XGBoost (No PCA),0.679,0.673,0.677,0.765,0.718,0.703


In [ ]:
# exporting to latex
def create_latex_table(
    best_df ,
    caption,
    label,
):
    table = best_df[
        [
            "language",
            "features",

            "model_display" ,
            "accuracy",
            "balanced_accuracy",
            "precision",

            "recall",
            "f1",
            "auc",
        ]
    ].copy()

    table = table.rename(
        columns={
            "language": "Language",
            "features": "Features",
            "model_display": "Model",
            "accuracy": "Accuracy",
            "balanced_accuracy": "Balanced accuracy",
            "precision": "Precision",
            "recall": "Recall",
            "f1": "F1-score",
            "auc": "AUC",
        }
    )

    table["Language"] = pd.Categorical(
        table["Language"],categories=LANGUAGE_ORDER,  ordered=True )

    table["Features"] = pd.Categorical(table["Features"],categories=FEATURE_ORDER,ordered=True)

    table = (
        table.sort_values(["Language","Features",]).set_index(["Language", "Features"]
        ))
    tabular = table.to_latex(
        index=True,
        escape=True,
        multirow=True,
        sparsify=True,
        column_format="lllcccccc",
        float_format="%.3f",
    )

    latex_code = rf"""
\begin{{table*}}[htbp]
\centering
\caption{{{caption}}}
\label{{{label}}}
\resizebox{{\textwidth}}{{!}}{{%
{tabular}
}}
\end{{table*}}
"""

    return latex_code

mono_latex = create_latex_table(
    best_df=best_mono,
    caption=(
        "Monolingual classification performance. "
        "For each language and feature representation, "
        "the configuration with the highest balanced accuracy is reported."
    ),
    label="tab:mono_best_configurations",
)

cross_latex = create_latex_table(
    best_df=best_cross,
    caption=(
        "Cross-lingual classification performance. "
        "For each held-out language and feature representation, "
        "the configuration with the highest balanced accuracy is reported."
    ),
    label="tab:cross_best_configurations",
)

multi_latex = create_latex_table(
    best_df=best_multi,
    caption=(
        "Multilingual classification performance. "
        "For each test language and feature representation, "
        "the configuration with the highest balanced accuracy is reported."
    ),
    label="tab:multi_best_configurations",
)

print(mono_latex)
print(cross_latex)
print(multi_latex)
LATEX_OUTPUT_DIR = RESULTS_ROOT / "latex_tables"
LATEX_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

latex_outputs = {
    "mono_best_configurations.tex": mono_latex,
    "cross_best_configurations.tex": cross_latex,
    "multi_best_configurations.tex": multi_latex,
}

for filename, latex_code in latex_outputs.items():
    output_path = LATEX_OUTPUT_DIR / filename

    output_path.write_text(
        latex_code,
        encoding="utf-8",
    )

    print(f"Saved: {output_path}")